In [62]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from scipy.optimize import minimize
from helper import *
import pandas as pd

In [2]:
mu_sun= 1.327e11 ## km^3/s^2  
mu_earth= 3.986e5 ## km^3/s^2  # 3.986e5
mu_mars= 4.2830e4 ## km^3/s^2
mu_venus = 32.4776e4

mu_moon = 0.4902e4 ## km^3/s^2 Not given in paper

SOI_earth = 923502.24 # km
SOI_mars = 577723.87 # km 577723.87 
SOI_moon = 66300

D_mars = 2.279e8
D_earth = 1.496e8
D_venus = 1.0815e8
D_moon = 384400 # km

r_LEO = 463
R_LEO = 6378.2 + r_LEO

w_earth = 1.99177621e-7 # rad/s
w_mars = 1.05850987e-7 # rad/s
w_venus = 3.23861161e-7
w_moon = 2.6653e-6 # rad/s

r_LMO = 200
R_LMO = 3397.0 + r_LMO

In [3]:
def earth_mars_hohmann_transfer(result_str = False):
    mu_sun= 1.327e11 ## km^3/s^2
    mu_earth= 3.986e5 ## km^3/s^2
    mu_mars= 4.2830e4 ## km^3/s^2

    SOI_earth = 923502.24 # km
    SOI_mars = 577723.87 # km

    
    r_mars = 2.279e8
    r_earth = 1.496e8

    r_LEO = 463
    R_LEO = 6378.2 + r_LEO

    r_LMO = 200
    R_LMO = 3397.0 + r_LMO

    a_transfer = (r_mars + r_earth)/2


    delta_v1 = np.sqrt(mu_sun/r_earth) * (np.sqrt((2*r_mars)/(r_earth+r_mars)) - 1)

    delta_v2 = np.sqrt(mu_sun/r_mars) * (1 - np.sqrt((2*r_earth)/(r_earth+r_mars)))

    v_p = np.sqrt((delta_v1)**2 + ((2*mu_earth)/R_LEO))

    v_c = np.sqrt(mu_earth/R_LEO)
    delta_v_Earth = v_p - v_c
    delta_v_inf = np.sqrt(mu_sun/r_mars)*(1 - np.sqrt((2*r_earth)/(r_earth + r_mars)))
    delta_v_mars = np.sqrt(delta_v_inf**2 + (2*mu_mars/R_LMO)) - np.sqrt(mu_mars*(1)/R_LMO)

    delta_v_total = delta_v_Earth + delta_v_mars


    t_of_f = np.pi * np.sqrt(((r_mars + r_earth)**3)/(8*mu_sun))
    t_of_f_days = t_of_f/(60*60*24)

    
    

    if result_str:
        print(f"""
Delta V_Leo = {delta_v_Earth}
Delta V_LMO = {delta_v_mars}
Delta V_Total = {delta_v_total}

Time of Flight = {t_of_f_days:.2f} Days
Semi-major axis = {a_transfer:e} Km
""")

    return delta_v_total

earth_mars_hohmann_transfer(result_str= True)



Delta V_Leo = 3.5558147563670417
Delta V_LMO = 2.101362221346033
Delta V_Total = 5.657176977713075

Time of Flight = 258.84 Days
Semi-major axis = 1.887500e+08 Km



np.float64(5.657176977713075)

In [53]:
def lambert_solve(R1,R2,t,mu, trajectory_type, debug = False):
    # t in days
    t = t* (60*60*24) # convert to seconds


    r1 = np.linalg.norm(R1) # Mag of R1
    r2 = np.linalg.norm(R2) # Mag of R2

    cross_r1_r2 = np.cross(R1,R2) # cross product if R1 and R2

    if trajectory_type == "pro": # (3.23)
        if cross_r1_r2[2] >= 0:
            theta = np.arccos(np.dot(R1,R2)/(r1*r2))
        
        else:
            theta = 2*np.pi - np.arccos(np.dot(R1,R2)/(r1*r2))

    elif trajectory_type == "retro": # (3.24)
        if cross_r1_r2[2] < 0:
            theta = np.arccos(np.dot(R1,R2)/(r1*r2))
        
        else:
            theta = 2*np.pi - np.arccos(np.dot(R1,R2)/(r1*r2))

    else:
        print("Trajectory type not specified")
        return

    # print(np.rad2deg(theta))
    # if np.rad2deg(theta) == 180.0:
    #     print("180deg anomaly => Hohmann Transfer")
    #     print(f"{earth_mars_hohmann_transfer()}")
    #     return earth_mars_hohmann_transfer() 
    A = np.sin(theta) * np.sqrt((r1*r2) / (1-np.cos(theta))) # (3.25)

    ## Determine where approximately F(z) changes sign. This will be used as an initial guess for root-finding.
    z = -100
    while F(z,t,r1,r2,mu,A) < 0:
        z = z + 0.1


    ## Error Tolerance and max number of iterations


    # Apply Newton-Raphson root-finder from scipy library
    z = optimize.newton(F,z, args = (t,r1,r2,mu,A)) 
    print(z)

    # Lagrange Coefficients
    f = 1 - (y(z,r1,r2,A)/r1) # (3.31)
    g = A*np.sqrt(y(z,r1,r2,A)/mu) # (3.32)

    
    fdot = (np.sqrt(mu)/(r1*r2)) * (np.sqrt(y(z,r1,r2,A)/stumpC(z)) * (z*stumpS(z) -1)) # (3.33)
    gdot = 1 - (y(z,r1,r2,A)/r2) # (3.34)
    
    if debug:
        print(f"{g = } {R2 = } {f = } {R1 = }")
    V1 = (1/g)*(R2 - f*R1) # (3.35)
    
    V2 = (1/g)*(gdot*R2 - R1) # (3.36)

    if debug:
        print(f"{r1 = }\t{r2 = }\t{cross_r1_r2 = }\t{A = } \n {V1 = }\t{V2 = }")

    return V1, V2


mu_sun = 1.327e11  # Suns gravitational parameter in (km^3/s^2) 

R1 = np.array([1.4762e+08 ,0.0000e+00 ,0.0000e+00])
R2 = np.array([-2.2800000e+08 , 2.7921947e-08 , 0.0000000e+00])
t_of_f = 60*60*24 * 270
lambert_solve(R1,R2, t_of_f, mu_sun, "pro", debug = True)

38.7769660198383
g = np.float64(8.452297419925728e-10) R2 = array([-2.2800000e+08,  2.7921947e-08,  0.0000000e+00]) f = np.float64(-1.544506164476358) R1 = array([1.4762e+08, 0.0000e+00, 0.0000e+00])
r1 = np.float64(147620000.0)	r2 = np.float64(228000000.0)	cross_r1_r2 = array([ 0.        , -0.        ,  4.12183782])	A = np.float64(1.588677986858443e-08) 
 V1 = array([-35.2594341 ,  33.03474264,   0.        ])	V2 = array([  0.        , -21.38854697,  -0.        ])


(array([-35.2594341 ,  33.03474264,   0.        ]),
 array([  0.        , -21.38854697,  -0.        ]))

In [166]:
def simulate(anomaly, t_of_f): # t_of_f in degrees
    if anomaly == 180:
        return earth_mars_hohmann_transfer()
    R_earth = 1.4960e8 # km, average distance from sun
    h_earth = np.sqrt(R_earth * mu_sun) # circular orbit
    earth_orbit = Orbit(h_earth,0,0,0,0,0) # circular orbit

    R1, V1 = calculate_state_vector_from_orbit(earth_orbit,mu_sun,debug= False)

    R_mars = 2.279e8 # km, average distance from sun
    h_mars = np.sqrt(R_mars * mu_sun)
    mars_orbit = Orbit(h_mars,0,0,0,0,anomaly) # Weird Vd calculated at 180 deg. 
    print(anomaly)
    R2, V2 = calculate_state_vector_from_orbit(mars_orbit,mu_sun,debug=False)
    # print(f"{R2} {V2}")

    Vd, Va = lambert_solve(np.array(R1),np.array(R2),t_of_f,mu_sun,"pro", debug = False)
    print(f"{Vd = } {Va = }")
    v_inf_d = Vd - V1
    v_inf_a = Va - V2
    print(f"{v_inf_d = } {v_inf_a = }")

    r_LEO = 6378.2 + 463
    v_inf_d_mag = np.linalg.norm(v_inf_d)
    v_periapse = np.sqrt((v_inf_d_mag**2) + (2*mu_earth/r_LEO))
    v_o = np.sqrt(mu_earth/r_LEO)
    delta_v1 = v_periapse - v_o

    r_LMO = 3397 + 200
    v_inf_a_mag = np.linalg.norm(v_inf_a)
    v_periapse = np.sqrt((v_inf_a_mag**2) + (2*mu_mars/r_LMO))
    v_o = np.sqrt(mu_mars/r_LMO)

    delta_v2 = v_periapse - v_o
    delta_total = delta_v1 + delta_v2
    print(delta_v1, delta_v2, delta_total)
    return delta_v1, delta_v2, delta_total



## Given Anomaly

In [92]:
simulate(182,402)

14.019822029948564
Vd = array([ 7.31748344, 32.80464834, -0.        ]) Va = array([  8.26115987, -21.25853797,   0.        ])
v_inf_d = array([ 7.31748344,  3.02156446, -0.        ]) v_inf_a = array([7.41902342, 2.85709457, 0.        ])


(np.float64(5.753607938259554),
 np.float64(5.877734461865707),
 np.float64(11.631342400125261))

In [66]:
initial_guess = 300

def wrapped_simulate(x, anomaly):
    t_of_f = x
    deltaV_LEO, deltaV_LMO, delta_total = simulate(anomaly, t_of_f)
    return delta_total

optimized = minimize(wrapped_simulate,initial_guess, args = 185, method= "L-BFGS-B", bounds=[(200.0,400.0)])
print(f"\noptimized time = {optimized.x[0]} days")

[11.5825106]
Vd = array([ 2.03798596, 32.78666394, -0.        ]) Va = array([  4.39595505, -21.21970736,   0.        ])
v_inf_d = array([ 2.03798596,  3.00358006, -0.        ]) v_inf_a = array([2.29285804, 2.81880153, 0.        ])
[11.5825106]
Vd = array([ 2.03798596, 32.78666394, -0.        ]) Va = array([  4.39595505, -21.21970736,   0.        ])
v_inf_d = array([ 2.03798596,  3.00358006, -0.        ]) v_inf_a = array([2.29285804, 2.81880153, 0.        ])
[11.58136216]
Vd = array([ 2.0357583, 32.7866051, -0.       ]) Va = array([  4.39373162, -21.21986311,   0.        ])
v_inf_d = array([ 2.0357583 ,  3.00352122, -0.        ]) v_inf_a = array([2.29063461, 2.81864578, 0.        ])
[11.58136216]
Vd = array([ 2.0357583, 32.7866051, -0.       ]) Va = array([  4.39373162, -21.21986311,   0.        ])
v_inf_d = array([ 2.0357583 ,  3.00352122, -0.        ]) v_inf_a = array([2.29063461, 2.81864578, 0.        ])
[10.10460797]
Vd = array([-0.93496512, 32.70823562, -0.        ]) Va = array([  

In [ ]:
result = []

for anomaly in np.linspace(90,270,180):
    try:
        optimized = minimize(wrapped_simulate,initial_guess, args=anomaly, method = "Nelder-Mead", bounds=[(10,700)])
        result.append(anomaly)
        result.append(optimized.x[0])
    
    except:
        print(f"{anomaly} not valid")

[8.26921033]
Vd = array([20.59241939, 24.28136473,  0.        ]) Va = array([-15.93897395, -12.25002861,  -0.        ])
v_inf_d = array([20.59241939, -5.50171916,  0.        ]) v_inf_a = array([  8.19135814, -12.25002861,  -0.        ])
[8.62859145]
Vd = array([21.20008003, 24.00241866,  0.        ]) Va = array([-15.75586587, -12.95352724,  -0.        ])
v_inf_d = array([21.20008003, -5.78066522,  0.        ]) v_inf_a = array([  8.37446622, -12.95352724,  -0.        ])
[7.89187377]
Vd = array([19.93005437, 24.59059029,  0.        ]) Va = array([-16.14195835, -11.48142242,  -0.        ])
v_inf_d = array([19.93005437, -5.19249359,  0.        ]) v_inf_a = array([  7.98837374, -11.48142242,  -0.        ])
[7.49503845]
Vd = array([19.20485016, 24.93542556,  0.        ]) Va = array([-16.36831796, -10.63774257,  -0.        ])
v_inf_d = array([19.20485016, -4.84765833,  0.        ]) v_inf_a = array([  7.76201413, -10.63774257,  -0.        ])
[6.6358195]
Vd = array([17.52420497, 25.7603404 ,  0

In [69]:
result_shaped = np.reshape(result, (-1, 2))
result_shaped
df = pd.DataFrame(result_shaped)
with pd.ExcelWriter("output.xlsx") as writer:
    df.to_excel(writer, sheet_name="Sheet_name_1")

In [73]:
df.columns = ["anomaly", "time of flight"]
df

,anomaly,time of flight
0,90.000000,136.082439
1,91.005587,137.437935
2,92.011173,138.795147
3,93.016760,140.154018
4,94.022346,141.514492
...,...,...
175,265.977654,362.660637
176,266.983240,363.557568
177,267.988827,364.436874
178,268.994413,365.297699


In [74]:
delta_vs = []
for index, row in df.iterrows():
    res = simulate(row["anomaly"], row["time of flight"])
    delta_vs.append(res)
    

2.7487762398556335
Vd = array([ 7.161458  , 31.70770077,  0.        ]) Va = array([-20.81383078,   3.73241199,   0.        ])
v_inf_d = array([7.161458  , 1.92461689, 0.        ]) v_inf_a = array([3.31650131, 3.73241199, 0.        ])
2.802154408096544
Vd = array([ 7.0434274 , 31.74183146,  0.        ]) Va = array([-20.89747683,   3.30618786,   0.        ])
v_inf_d = array([7.0434274 , 1.95874757, 0.        ]) v_inf_a = array([3.22913892, 3.72967268, 0.        ])
2.8561492513623925
Vd = array([ 6.9271342 , 31.77483408,  0.        ]) Va = array([-20.97185257,   2.8789497 ,   0.        ])
v_inf_d = array([6.9271342, 1.9917502, 0.       ]) v_inf_a = array([3.14361529, 3.72578889, 0.        ])
2.9107604297572522
Vd = array([ 6.81253065, 31.80675284,  0.        ]) Va = array([-21.03699005,   2.45088385,   0.        ])
v_inf_d = array([6.81253065, 2.02366895, 0.        ]) v_inf_a = array([3.05990181, 3.72081657, 0.        ])
2.9659875071599227
Vd = array([ 6.69957001, 31.83763002,  0.        

In [75]:
delta_vs_shaped = np.reshape(delta_vs, (-1, 3))
df = pd.DataFrame(delta_vs_shaped)
with pd.ExcelWriter("deltaVs.xlsx") as writer:
    df.to_excel(writer, sheet_name="Sheet_name_1")

In [78]:
df

,0,1,2
0,5.463433,3.531044,8.994477
1,5.404350,3.488497,8.892848
2,5.346916,3.447016,8.793932
3,5.291078,3.406569,8.697646
4,5.236786,3.367128,8.603914
...,...,...,...
175,5.236785,3.367128,8.603914
176,5.291079,3.406568,8.697646
177,5.346917,3.447015,8.793932
178,5.404351,3.488497,8.892848


In [79]:
df.columns = ["LEO", "LMO", "TOTAL"]

In [80]:
with pd.ExcelWriter("deltaVs.xlsx") as writer:
    df.to_excel(writer, sheet_name="Sheet_name_1")

## Given Time of Flight

In [ ]:
initial_guess = 160

def wrapped_simulate(x, t_of_f):
    anomaly = x
    deltaV_LEO, deltaV_LMO, delta_total = simulate(anomaly, t_of_f)
    return delta_total

optimized = minimize(wrapped_simulate,initial_guess, args=259, method = "Nelder-Mead", bounds=[(130,240)])
print(f"{optimized.x[0] = }")


[160.]
8.932641912597093
Vd = array([ 3.68253107, 32.44026584,  0.        ]) Va = array([ -5.6695144 , -20.59781958,  -0.        ])
v_inf_d = array([3.68253107, 2.65718196, 0.        ]) v_inf_a = array([ 2.58354524,  2.07727542, -0.        ])
4.078017629394747 2.4488242392463224 6.526841868641069
[168.]
9.279668759929645
Vd = array([ 2.16265344, 32.62671449,  0.        ]) Va = array([ -3.48990287, -21.1537663 ,  -0.        ])
v_inf_d = array([2.16265344, 2.84363061, 0.        ]) v_inf_a = array([ 1.52707528,  2.44926014, -0.        ])
3.737561749579804 2.2189938939226304 5.956555643502434
[176.]
9.666513218966049
Vd = array([ 0.71160198, 32.71554582,  0.        ]) Va = array([ -1.17973832, -21.44535417,  -0.        ])
v_inf_d = array([0.71160198, 2.93246194, 0.        ]) v_inf_a = array([ 0.50350855,  2.62619764, -0.        ])
3.5755735579549572 2.1139225358324945 5.689496093787452
[184.]
10.09586629563254
Vd = array([-0.66765765, 32.71647208, -0.        ]) Va = array([  1.22362911, -2

TypeError: cannot unpack non-iterable numpy.float64 object

In [132]:
wrapped_simulate(1,160)

1
4.015343851784693
Vd = array([26.8133971,  0.4152297,  0.       ]) Va = array([-10.46920653,   0.08986935,   0.        ])
v_inf_d = array([ 26.8133971 , -29.36785418,   0.        ]) v_inf_a = array([-10.04807417, -24.03678757,   0.        ])


np.float64(56.62806024639516)

In [136]:
result = []

for t_of_f in range(200,400, 1):
    try:
        optimized = minimize(wrapped_simulate,initial_guess, args=t_of_f, method = "Nelder-Mead", bounds=[(0,359)])
        result.append(t_of_f)
        result.append(optimized.x[0])
    
    except:
        print(f"{t_of_f} not valid")

[160.]
6.449572874401607
Vd = array([-2.23507406, 33.07278698,  0.        ]) Va = array([-11.4082605 , -18.95093849,  -0.        ])
v_inf_d = array([-2.23507406,  3.2897031 ,  0.        ]) v_inf_a = array([-3.15520086,  3.72415651, -0.        ])
[168.]
6.720452352080005
Vd = array([-3.82250818, 33.00797232,  0.        ]) Va = array([ -9.40977477, -20.15131824,  -0.        ])
v_inf_d = array([-3.82250818,  3.22488844,  0.        ]) v_inf_a = array([-4.39279662,  3.45170819, -0.        ])
[152.]
6.216644120718839
Vd = array([-0.54862208, 33.02322213,  0.        ]) Va = array([-13.15903114, -17.55436607,  -0.        ])
v_inf_d = array([-0.54862208,  3.24013825,  0.        ]) v_inf_a = array([-1.83052642,  3.75145256, -0.        ])
[144.]
6.018174913120597
Vd = array([ 1.23017266, 32.84492844,  0.        ]) Va = array([-14.64394864, -16.01059335,  -0.        ])
v_inf_d = array([1.23017266, 3.06184456, 0.        ]) v_inf_a = array([-0.46049531,  3.51125539, -0.        ])
[128.]
5.7109108596

In [138]:
result_shaped = np.reshape(result, (-1, 2))
result_shaped
df = pd.DataFrame(result_shaped)
df.columns = ["time of flight", "anomaly"]
with pd.ExcelWriter("output.xlsx") as writer:
    df.to_excel(writer, sheet_name="Sheet_name_1")

In [142]:
delta_vs = []
for index, row in df.iterrows():
    res = simulate(row["anomaly"], row["time of flight"])
    delta_vs.append(res)
    

140.52252197265625
5.941773719167402
Vd = array([ 2.02990192, 32.72362725,  0.        ]) Va = array([-15.20390577, -15.30618221,  -0.        ])
v_inf_d = array([2.02990192, 2.94054337, 0.        ]) v_inf_a = array([ 0.13755271,  3.31940713, -0.        ])
141.21533203125
6.001710326114953
Vd = array([ 1.9898984 , 32.72405173,  0.        ]) Va = array([-14.98943787, -15.51198542,  -0.        ])
v_inf_d = array([1.9898984 , 2.94096784, 0.        ]) v_inf_a = array([ 0.12568735,  3.29774388, -0.        ])
141.90789794921875
6.061919412395191
Vd = array([ 1.95006809, 32.72446185,  0.        ]) Va = array([-14.77241906, -15.71465832,  -0.        ])
v_inf_d = array([1.95006809, 2.94137796, 0.        ]) v_inf_a = array([ 0.11424383,  3.27639733, -0.        ])
142.60009765625
6.122397729220296
Vd = array([ 1.91043442, 32.72485342,  0.        ]) Va = array([-14.55291378, -15.91415012,  -0.        ])
v_inf_d = array([1.91043442, 2.94176954, 0.        ]) v_inf_a = array([ 0.10323426,  3.25536347, 

In [143]:
delta_vs_shaped = np.reshape(delta_vs, (-1, 3))
df = pd.DataFrame(delta_vs_shaped)
df.columns = ["LEO", "LMO", "TOTAL"]
with pd.ExcelWriter("deltaVs.xlsx") as writer:
    df.to_excel(writer, sheet_name="Sheet_name_1")

## Opt both

In [167]:
initial_guess = [170,200]

def wrapped_simulate(x):
    anomaly, t_of_f = x
    deltaV_LEO, deltaV_LMO, delta_total  = simulate(anomaly, t_of_f)
    return delta_total

optimized = minimize(wrapped_simulate,initial_guess, bounds=((1,360),(200,400)))
print(f"optimized angle = {optimized.x[0]} degrees    \noptimized time = {optimized.x[1]} days")

170.0
6.7945123689926135
Vd = array([-4.20335879, 32.97553758,  0.        ]) Va = array([ -8.87444277, -20.41519658,  -0.        ])
v_inf_d = array([-4.20335879,  3.19245369,  0.        ]) v_inf_a = array([-4.68425457,  3.34854154, -0.        ])
4.383080515774298 4.097125322710163 8.48020583848446
170.00000001
6.7945123693694836
Vd = array([-4.20335879, 32.97553758,  0.        ]) Va = array([ -8.87444276, -20.41519658,  -0.        ])
v_inf_d = array([-4.20335879,  3.19245369,  0.        ]) v_inf_a = array([-4.68425457,  3.34854154, -0.        ])
4.383080516387553 4.0971253233586316 8.480205839746183
170.0
6.7945123695105965
Vd = array([-4.20335879, 32.97553758,  0.        ]) Va = array([ -8.87444276, -20.41519658,  -0.        ])
v_inf_d = array([-4.20335879,  3.19245369,  0.        ]) v_inf_a = array([-4.68425457,  3.34854154, -0.        ])
4.383080515303426 4.097125321833279 8.480205837136705
169.87382778305158
6.796736890744781
Vd = array([-4.16210036, 32.97682543,  0.        ]) Va =

In [168]:
simulate(optimized.x[0], optimized.x[1])

180.30891755341997
9.906804777968665
Vd = array([-9.80545310e-03,  3.27264172e+01, -0.00000000e+00]) Va = array([  0.13633105, -21.48212224,   0.        ])
v_inf_d = array([-0.00980545,  2.94333333, -0.        ]) v_inf_a = array([0.0062299 , 2.64785912, 0.        ])
3.5558213434555483 2.101397363990827 5.657218707446376


(np.float64(3.5558213434555483),
 np.float64(2.101397363990827),
 np.float64(5.657218707446376))